In [1]:
import polars as pl
import httpx
from dotenv import load_dotenv
import os

load_dotenv()

PURPLE_AIR_API_KEY = os.getenv("PURPLE_AIR_API_KEY")

In [2]:
# get json data from purple air response - note this code is from httpx documentation and just modified to work with this project's usecase

try:
    sensors = httpx.get(
        "https://api.purpleair.com/v1/sensors",
        params={
            "api_key": PURPLE_AIR_API_KEY,
            "fields": "latitude,longitude,position_rating,location_type,last_seen",
            "max_age": 0,
            "nwlat": 40.962891,
            "nwlng": -74.264395,
            "selat": 40.470116,
            "selng": -73.675450,
        },
    )
    if sensors.status_code == 200:
        sensors_json = sensors.json()
        print(sensors_json)
    else:
        print(f"Request failed with status code: {sensors.status_code}, response: {sensors.text}")
except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
except httpx.RequestError as e:
    print(f"An error occurred while trying to make the request: {e}")

{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1774201740, 'data_time_stamp': 1774201710, 'max_age': 0, 'firmware_default_version': '7.02', 'fields': ['sensor_index', 'last_seen', 'location_type', 'position_rating', 'latitude', 'longitude'], 'location_types': ['outside', 'inside'], 'data': [[262999, 1774201640, 1, 0, 40.801434, -73.970535], [263287, 1755525360, 1, 5, 40.842293, -74.07955], [263301, 1755525237, 1, 5, 40.842327, -74.079544], [263457, 1755525786, 1, 5, 40.84218, -74.07933], [1660, 1663947634, 0, 5, 40.721, -74.1928], [2319, 1666785882, 0, 5, 40.809963, -73.95416], [264499, 1762224167, 1, 5, 40.716164, -73.96324], [2897, 1518323822, 0, 5, 40.790447, -73.733284], [265475, 1774147551, 1, 0, 40.682373, -73.97324], [265803, 1773728621, 0, 0, 40.686188, -73.94205], [4803, 1774201661, 0, 5, 40.787937, -73.96807], [5100, 1511917657, 1, 0, 40.734493, -73.8193], [5110, 1511921460, 1, 0, 40.735878, -73.85171], [5444, 1533082789, 0, 0, 40.844254, -73.98846], [5476, 1527782528, 1, 5, 

In [3]:
raw_sensors_df = pl.DataFrame(data = sensors_json["data"], \
                  schema = sensors_json["fields"])

print(raw_sensors_df.height)
raw_sensors_df.head()

595


C:\Users\crist\AppData\Local\Temp\ipykernel_34068\315921795.py:1: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  raw_sensors_df = pl.DataFrame(data = sensors_json["data"], \


sensor_index,last_seen,location_type,position_rating,latitude,longitude
i64,i64,i64,i64,f64,f64
262999,1774201640,1,0,40.801434,-73.970535
263287,1755525360,1,5,40.842293,-74.07955
263301,1755525237,1,5,40.842327,-74.079544
263457,1755525786,1,5,40.84218,-74.07933
1660,1663947634,0,5,40.721,-74.1928


### Cleaning data

Since we are looking to create a dashboard that shows air quality in NYC, we need to firstly clean the data and do a spatial join to get sensors only within the 5 boroughs,
and also remove the sensors which are in indoors (location_type = 1) and those with a position rating of 0 (position_rating = 0) since they are likely to be inaccurate. We will also convert the last_seen column to a datetime format.

In [9]:
import geopandas as gpd

raw_sensors_gdf = gpd.GeoDataFrame(
                    raw_sensors_df.to_pandas(),
                    geometry=gpd.points_from_xy(raw_sensors_df["longitude"], raw_sensors_df["latitude"]),
                    crs="EPSG:4326"
                )

raw_sensors_gdf = raw_sensors_gdf.loc[(raw_sensors_gdf["position_rating"] > 0) & (raw_sensors_gdf["location_type"] != 1)]

raw_sensors_gdf.head()

,sensor_index,last_seen,location_type,position_rating,latitude,longitude,geometry
4,1660,1663947634,0,5,40.721000,-74.192800,POINT (-74.1928 40.721)
5,2319,1666785882,0,5,40.809963,-73.954160,POINT (-73.95416 40.80996)
7,2897,1518323822,0,5,40.790447,-73.733284,POINT (-73.73328 40.79045)
10,4803,1774201661,0,5,40.787937,-73.968070,POINT (-73.96807 40.78794)
15,5648,1555619294,0,5,40.735424,-74.009460,POINT (-74.00946 40.73542)


In [25]:
# Now that the data is in a geodataframe, we can do a spatial join to remove sensors outside of the 5 boroughs of NYC.
# The borough boundaries data is in the NYC Open Data portal.

response = httpx.get("https://data.cityofnewyork.us/resource/gthc-hcne.geojson")
response.raise_for_status()

geojson = response.json()

boroughs_gdf = gpd.GeoDataFrame.from_features(geojson["features"], crs="EPSG:4326")
boroughs_gdf.head()


,geometry,borocode,boroname,shape_area,shape_leng
0,"MULTIPOLYGON (((-74.05051 40.56642, -74.05047 ...",5,Staten Island,1623618358.46,325912.288988
1,"MULTIPOLYGON (((-74.01093 40.68449, -74.01193 ...",1,Manhattan,636631650.451,359537.866079
2,"MULTIPOLYGON (((-73.89681 40.79581, -73.89694 ...",2,Bronx,1187199300.36,463147.071867
3,"MULTIPOLYGON (((-73.86327 40.58388, -73.86381 ...",3,Brooklyn,1934462607.43,726953.044632
4,"MULTIPOLYGON (((-73.82645 40.59053, -73.82642 ...",4,Queens,3041419178.99,887905.076018
